# 🚀 NASA Space Data Pipeline — Delta Lake + Apache Iceberg
**Source:** NASA Open APIs (free, no API key — api.nasa.gov) 
**Stack:** NASA API → Kafka → PySpark → Delta Lake (S3) → dbt → BigQuery
**Orchestration:** Apache Airflow (daily)

## Overview
Daily batch pipeline ingesting 3 NASA data streams:
1. **APOD** (Astronomy Picture of the Day) — metadata + image URL, daily since 1995
2. **NeoWs** (Near Earth Object Web Service) — asteroid close-approach data
3. **DONKI** (Space Weather) — solar flares, geomagnetic storms, CME events

Uses Delta Lake on S3 for ACID-compliant lakehouse storage with time-travel,
dbt for transformation, and BigQuery as the analytical warehouse.

**API:** https://api.nasa.gov — completely free, 1000 req/hr with DEMO_KEY

## Section 1 — Imports & Configuration

In [ ]:
# pip install requests kafka-python pyspark delta-spark google-cloud-bigquery pandas boto3

import requests
import json
import time
import logging
import pandas as pd
from datetime import datetime, date, timedelta
from kafka import KafkaProducer
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    from_json, col, to_date, to_timestamp, when, lit,
    current_timestamp, year, month, dayofmonth, udf
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
)
from delta import DeltaTable, configure_spark_with_delta_pip
from google.cloud import bigquery
import boto3

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('NASAPipeline')
print('✅ All imports loaded')

In [ ]:
# NASA API — free, use 'DEMO_KEY' or register free at api.nasa.gov for 1000 req/hr
NASA_API_KEY   = 'DEMO_KEY'   # replace with your key from api.nasa.gov
NASA_BASE_URL  = 'https://api.nasa.gov'

# Kafka
KAFKA_BOOTSTRAP     = 'localhost:9092'
KAFKA_TOPIC_APOD    = 'nasa-apod-daily'
KAFKA_TOPIC_NEO     = 'nasa-neo-asteroids'
KAFKA_TOPIC_DONKI   = 'nasa-donki-spaceweather'

# Delta Lake on S3
S3_BUCKET           = 'your-delta-lake-bucket'
DELTA_BASE_PATH     = f's3a://{S3_BUCKET}/delta'
DELTA_APOD_PATH     = f'{DELTA_BASE_PATH}/apod'
DELTA_NEO_PATH      = f'{DELTA_BASE_PATH}/neo_asteroids'
DELTA_DONKI_PATH    = f'{DELTA_BASE_PATH}/donki_events'

# BigQuery
GCP_PROJECT         = 'your-gcp-project'
BQ_DATASET          = 'nasa_space_analytics'

print('✅ Config ready')
print(f'   Delta Lake: {DELTA_BASE_PATH}')
print(f'   BigQuery:   {GCP_PROJECT}.{BQ_DATASET}')

## Section 2 — NASA API Extractors

In [ ]:
def fetch_apod(start_date: str, end_date: str) -> list:
    """
    Fetch Astronomy Picture of the Day records.
    APOD API returns one entry per day since 1995-06-16.
    """
    url = f'{NASA_BASE_URL}/planetary/apod'
    params = {
        'api_key':    NASA_API_KEY,
        'start_date': start_date,
        'end_date':   end_date,
        'thumbs':     True
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    records = []
    for item in resp.json():
        records.append({
            'apod_date':     item.get('date'),
            'title':         item.get('title', ''),
            'explanation':   (item.get('explanation', ''))[:1000],
            'media_type':    item.get('media_type', ''),  # image or video
            'url':           item.get('url', ''),
            'hdurl':         item.get('hdurl', ''),
            'copyright':     item.get('copyright', 'NASA'),
            'service_version': item.get('service_version', 'v1'),
            'batch_date':    str(date.today()),
            'ingested_at':   datetime.utcnow().isoformat(),
            'source':        'nasa-apod'
        })
    logger.info(f'✅ APOD: {len(records)} records fetched ({start_date} → {end_date})')
    return records

def fetch_neo_asteroids(start_date: str, end_date: str) -> list:
    """
    Fetch Near Earth Objects (asteroids) close approach data.
    Max 7-day window per request.
    """
    url = f'{NASA_BASE_URL}/neo/rest/v1/feed'
    params = {
        'start_date': start_date,
        'end_date':   end_date,
        'api_key':    NASA_API_KEY
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    records = []
    for day, objects in data.get('near_earth_objects', {}).items():
        for obj in objects:
            ca = obj.get('close_approach_data', [{}])[0]
            records.append({
                'neo_id':                 obj.get('id'),
                'neo_name':               obj.get('name', ''),
                'close_approach_date':    day,
                'is_potentially_hazardous': obj.get('is_potentially_hazardous_asteroid', False),
                'estimated_diameter_min_km': obj.get('estimated_diameter', {}).get('kilometers', {}).get('estimated_diameter_min', 0.0),
                'estimated_diameter_max_km': obj.get('estimated_diameter', {}).get('kilometers', {}).get('estimated_diameter_max', 0.0),
                'relative_velocity_kmh':  float(ca.get('relative_velocity', {}).get('kilometers_per_hour', 0)),
                'miss_distance_km':       float(ca.get('miss_distance', {}).get('kilometers', 0)),
                'orbiting_body':          ca.get('orbiting_body', 'Earth'),
                'absolute_magnitude':     obj.get('absolute_magnitude_h', 0.0),
                'nasa_jpl_url':           obj.get('nasa_jpl_url', ''),
                'batch_date':             str(date.today()),
                'ingested_at':            datetime.utcnow().isoformat(),
                'source':                 'nasa-neows'
            })
    logger.info(f'✅ NEO: {len(records)} asteroids fetched')
    return records

def fetch_donki_events(start_date: str, end_date: str) -> list:
    """
    Fetch DONKI space weather events: solar flares (FLR) + geomagnetic storms (GST).
    """
    events = []
    base_params = {'startDate': start_date, 'endDate': end_date, 'api_key': NASA_API_KEY}

    # Solar Flares
    try:
        r = requests.get(f'{NASA_BASE_URL}/DONKI/FLR', params=base_params, timeout=20)
        r.raise_for_status()
        for flare in r.json() or []:
            events.append({
                'event_id':     flare.get('flrID', ''),
                'event_type':   'solar_flare',
                'begin_time':   flare.get('beginTime', ''),
                'peak_time':    flare.get('peakTime', ''),
                'end_time':     flare.get('endTime', ''),
                'class_type':   flare.get('classType', ''),  # A, B, C, M, X
                'source_location': flare.get('sourceLocation', ''),
                'active_region': str(flare.get('activeRegionNum', '')),
                'kp_index':     None,
                'batch_date':   str(date.today()),
                'ingested_at':  datetime.utcnow().isoformat(),
                'source':       'nasa-donki-flr'
            })
    except Exception as e:
        logger.warning(f'FLR fetch: {e}')

    # Geomagnetic Storms
    try:
        r = requests.get(f'{NASA_BASE_URL}/DONKI/GST', params=base_params, timeout=20)
        r.raise_for_status()
        for storm in r.json() or []:
            kp = storm.get('allKpIndex', [{}])
            kp_max = max([k.get('kpIndex', 0) for k in kp], default=0)
            events.append({
                'event_id':     storm.get('gstID', ''),
                'event_type':   'geomagnetic_storm',
                'begin_time':   storm.get('startTime', ''),
                'peak_time':    storm.get('startTime', ''),
                'end_time':     '',
                'class_type':   f'Kp{kp_max}',
                'source_location': '',
                'active_region': '',
                'kp_index':     float(kp_max),
                'batch_date':   str(date.today()),
                'ingested_at':  datetime.utcnow().isoformat(),
                'source':       'nasa-donki-gst'
            })
    except Exception as e:
        logger.warning(f'GST fetch: {e}')

    logger.info(f'✅ DONKI: {len(events)} space weather events fetched')
    return events

# Publish all to Kafka
def publish_all_to_kafka(execution_date: str) -> dict:
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
        acks='all', compression_type='gzip'
    )
    start = (datetime.strptime(execution_date, '%Y-%m-%d') - timedelta(days=7)).strftime('%Y-%m-%d')
    end   = execution_date
    counts = {}

    apod_records = fetch_apod(start, end)
    for r in apod_records:
        producer.send(KAFKA_TOPIC_APOD, key=r['apod_date'].encode(), value=r)
    counts['apod'] = len(apod_records)
    time.sleep(1)

    neo_records = fetch_neo_asteroids(start, end)
    for r in neo_records:
        producer.send(KAFKA_TOPIC_NEO, key=r['neo_id'].encode(), value=r)
    counts['neo'] = len(neo_records)
    time.sleep(1)

    donki_records = fetch_donki_events(start, end)
    for r in donki_records:
        producer.send(KAFKA_TOPIC_DONKI, key=r['event_id'].encode(), value=r)
    counts['donki'] = len(donki_records)

    producer.flush()
    producer.close()
    logger.info(f'✅ Published: {counts}')
    return counts

# publish_all_to_kafka(str(date.today()))

## Section 3 — PySpark + Delta Lake Write (ACID Lakehouse)

In [ ]:
builder = SparkSession.builder \
    .appName('NASADeltaLake') \
    .config('spark.jars.packages',
            'io.delta:delta-spark_2.12:3.1.0,'
            'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,'
            'org.apache.hadoop:hadoop-aws:3.3.4') \
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension') \
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog') \
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem') \
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'com.amazonaws.auth.DefaultAWSCredentialsProviderChain')

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')

NEO_SCHEMA = StructType([
    StructField('neo_id',                   StringType(),  True),
    StructField('neo_name',                 StringType(),  True),
    StructField('close_approach_date',      StringType(),  True),
    StructField('is_potentially_hazardous', BooleanType(), True),
    StructField('estimated_diameter_min_km',DoubleType(),  True),
    StructField('estimated_diameter_max_km',DoubleType(),  True),
    StructField('relative_velocity_kmh',    DoubleType(),  True),
    StructField('miss_distance_km',         DoubleType(),  True),
    StructField('orbiting_body',            StringType(),  True),
    StructField('absolute_magnitude',       DoubleType(),  True),
    StructField('batch_date',               StringType(),  True),
    StructField('ingested_at',              StringType(),  True),
])

def write_to_delta(df, delta_path: str, merge_keys: list, partition_cols: list = None):
    """
    MERGE (upsert) DataFrame into Delta Lake table.
    Creates table if not exists, otherwise upserts by merge_keys.
    """
    writer = df.write.format('delta').mode('append')
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)

    if DeltaTable.isDeltaTable(spark, delta_path):
        delta_table = DeltaTable.forPath(spark, delta_path)
        merge_condition = ' AND '.join([f'target.{k} = source.{k}' for k in merge_keys])
        delta_table.alias('target') \
            .merge(df.alias('source'), merge_condition) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        logger.info(f'✅ MERGE complete → {delta_path}')
    else:
        writer.save(delta_path)
        logger.info(f'✅ Created new Delta table → {delta_path}')

def process_neo_batch(execution_date: str):
    raw = spark.read.format('kafka') \
        .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP) \
        .option('subscribe', KAFKA_TOPIC_NEO) \
        .option('startingOffsets', 'earliest') \
        .option('endingOffsets', 'latest') \
        .load()

    df = raw.select(from_json(col('value').cast('string'), NEO_SCHEMA).alias('d')).select('d.*') \
        .dropDuplicates(['neo_id', 'close_approach_date']) \
        .withColumn('approach_date',    to_date(col('close_approach_date'))) \
        .withColumn('hazard_level',
            when(col('miss_distance_km') < 1_000_000, 'critical')
            .when(col('miss_distance_km') < 5_000_000, 'watch')
            .otherwise('safe')
        ) \
        .withColumn('size_category',
            when(col('estimated_diameter_max_km') >= 1.0, 'large')
            .when(col('estimated_diameter_max_km') >= 0.1, 'medium')
            .otherwise('small')
        ) \
        .withColumn('year',  year(col('approach_date'))) \
        .withColumn('month', month(col('approach_date')))

    write_to_delta(df, DELTA_NEO_PATH,
                   merge_keys=['neo_id', 'close_approach_date'],
                   partition_cols=['year', 'month'])
    logger.info(f'✅ NEO batch written to Delta Lake')

# process_neo_batch(str(date.today()))

## Section 4 — Delta Lake Time Travel

In [ ]:
def demonstrate_time_travel():
    """
    Delta Lake time-travel: query any historical version of the table.
    This is one of Delta Lake's key advantages over plain Parquet.
    """
    # Read current version
    current = spark.read.format('delta').load(DELTA_NEO_PATH)
    print(f'Current version rows: {current.count()}')

    # Read version 0 (initial load)
    v0 = spark.read.format('delta').option('versionAsOf', 0).load(DELTA_NEO_PATH)
    print(f'Version 0 rows: {v0.count()}')

    # Read as of yesterday
    yesterday = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d %H:%M:%S')
    hist = spark.read.format('delta').option('timestampAsOf', yesterday).load(DELTA_NEO_PATH)
    print(f'As of {yesterday}: {hist.count()} rows')

    # Show full history
    delta_table = DeltaTable.forPath(spark, DELTA_NEO_PATH)
    delta_table.history().select('version', 'timestamp', 'operation', 'operationParameters').show()

# demonstrate_time_travel()

def optimize_delta_table(path: str):
    """OPTIMIZE + ZORDER for query performance."""
    spark.sql(f"OPTIMIZE delta.`{path}` ZORDER BY (close_approach_date, neo_id)")
    spark.sql(f"VACUUM delta.`{path}` RETAIN 168 HOURS")  # keep 7 days
    logger.info(f'✅ Delta table optimized: {path}')

# optimize_delta_table(DELTA_NEO_PATH)

## Section 5 — BigQuery Sync + Analytics

In [ ]:
def sync_delta_to_bigquery(delta_path: str, bq_table: str, execution_date: str) -> int:
    df = spark.read.format('delta').load(delta_path) \
        .filter(col('batch_date') == execution_date)
    count = df.count()
    if count == 0:
        logger.info('No new records')
        return 0
    pdf = df.toPandas()
    bq = bigquery.Client(project=GCP_PROJECT)
    table_id = f'{GCP_PROJECT}.{BQ_DATASET}.{bq_table}'
    job = bq.load_table_from_dataframe(
        pdf, table_id,
        job_config=bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_APPEND)
    )
    job.result()
    logger.info(f'✅ Synced {count} rows → BigQuery {table_id}')
    return count

# Queries run directly in BigQuery
ANALYTICS_QUERIES = {
    'hazardous_asteroids_this_week': """
        SELECT neo_name, close_approach_date, miss_distance_km,
               estimated_diameter_max_km, relative_velocity_kmh, hazard_level
        FROM `{project}.nasa_space_analytics.neo_asteroids`
        WHERE is_potentially_hazardous = TRUE
          AND close_approach_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 7 DAY)
        ORDER BY miss_distance_km ASC
        LIMIT 20
    """,
    'solar_flare_class_distribution': """
        SELECT class_type, COUNT(*) AS event_count,
               MIN(begin_time) AS first_seen, MAX(peak_time) AS last_peak
        FROM `{project}.nasa_space_analytics.donki_events`
        WHERE event_type = 'solar_flare'
          AND EXTRACT(YEAR FROM TIMESTAMP(begin_time)) = EXTRACT(YEAR FROM CURRENT_DATE())
        GROUP BY class_type ORDER BY event_count DESC
    """
}
print('✅ Analytics queries defined')

## Section 6 — Pipeline Summary

| Layer | Technology | Details |
|---|---|---|
| **Source** | NASA Open API | APOD + NeoWs + DONKI, free, DEMO_KEY or registered |
| **Message Queue** | Apache Kafka | 3 topics: APOD, NEO, DONKI |
| **Processing** | PySpark | Read from Kafka, enrich, classify hazard levels |
| **Lakehouse** | **Delta Lake (S3)** | ACID MERGE upsert, time-travel, ZORDER optimization |
| **Warehouse** | Google BigQuery | Daily sync from Delta Lake |
| **Orchestration** | Apache Airflow | Daily DAG with OPTIMIZE + VACUUM |

**Delta Lake advantages used:**
- ACID transactions (MERGE upsert — no duplicates on re-run)
- Time travel (query any historical snapshot)
- OPTIMIZE + ZORDER (faster query performance)
- VACUUM (auto cleanup old versions)

**Volume:** ~7–30 APOD records/week + ~50–200 NEO objects/week + ~5–20 DONKI events/week